In [88]:
!pip install openai


## DATA ANNOTATION:

In [89]:
import os
import ast
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('config.env')

True

In [90]:
base_dir = os.getcwd()
data_dir = os.path.join(base_dir, 'data')

In [91]:
random_state = 42
vac_sample_size = 100
sampling_vars = ['length_category','vax_year']

## Create separate dataframes for each vaccine
data = pd.read_csv(os.path.join(data_dir, 'sampled_data.csv'))
pfizer_data = data[data['vax_manu'].str.strip() == 'PFIZER\BIONTECH']
moderna_data = data[data['vax_manu'].str.strip() == 'MODERNA']
janssen_data = data[data['vax_manu'].str.strip() == 'JANSSEN']

pfizer_frac = 120/pfizer_data.shape[0]
moderna_frac = 120/moderna_data.shape[0]
janssen_frac = 120/janssen_data.shape[0]

## Sample 100 records for each vaccine
s_pfizer_data = pfizer_data.groupby(
    sampling_vars, group_keys=False
).apply(lambda x: x.sample(frac=pfizer_frac, random_state=random_state), include_groups=True)
fs_pfizer_data = s_pfizer_data.sample(n=vac_sample_size, random_state=random_state)

s_moderna_data = moderna_data.groupby(
    sampling_vars, group_keys=False
).apply(lambda x: x.sample(frac=moderna_frac, random_state=random_state), include_groups=True)
fs_moderna_data = s_moderna_data.sample(n=vac_sample_size, random_state=random_state)

s_janssen_data = janssen_data.groupby(
    sampling_vars, group_keys=False
).apply(lambda x: x.sample(frac=janssen_frac, random_state=random_state), include_groups=True)
fs_janssen_data = s_janssen_data.sample(n=vac_sample_size, random_state=random_state)

sampled_data = pd.concat([fs_pfizer_data, fs_moderna_data, fs_janssen_data], ignore_index=True)
sampled_data.vax_manu.value_counts()

/var/folders/k4/zx_j67kd21n6d03ptwmp8_vm0000gn/T/ipykernel_96700/201950040.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  s_pfizer_data = pfizer_data.groupby(
/var/folders/k4/zx_j67kd21n6d03ptwmp8_vm0000gn/T/ipykernel_96700/201950040.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  s_moderna_data = moderna_data.groupby(
/var/folders/k4/zx_j67kd21n6d03ptwmp8_vm0000gn/T/ipykernel_96700/201950040.py:

vax_manu
PFIZER\BIONTECH                             100
MODERNA                                     100
JANSSEN                                     100
Name: count, dtype: int64

In [92]:
sampled_data.to_excel(os.path.join(data_dir, 'sample_annotation_data.xlsx'), index=False)

In [93]:
sampled_data = pd.read_excel(os.path.join(data_dir, 'sample_annotation_data.xlsx'))

In [95]:
sampled_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   vaers_id         300 non-null    int64  
 1   vax_type         300 non-null    object 
 2   vax_manu         300 non-null    object 
 3   age_yrs          278 non-null    float64
 4   sex              300 non-null    object 
 5   vax_year         300 non-null    int64  
 6   symptom_text     300 non-null    object 
 7   symptom_list     300 non-null    object 
 8   num_symptoms     300 non-null    int64  
 9   report_length    300 non-null    int64  
 10  length_category  300 non-null    object 
dtypes: float64(1), int64(4), object(6)
memory usage: 25.9+ KB


In [94]:
sampled_data.head()

,vaers_id,vax_type,vax_manu,age_yrs,sex,vax_year,symptom_text,symptom_list,num_symptoms,report_length,length_category
0,1989431,COVID19,PFIZER\BIONTECH,48.0,F,2021,Starting late in the afternoon on the day of i...,"['Chest pain', 'Computerised tomogram thorax',...",13,104,Medium
1,2083323,COVID19,PFIZER\BIONTECH,47.0,F,2022,Hives started 16 days after 2nd Covid vaccine....,"['Biopsy breast abnormal', 'Breast cancer fema...",12,137,Medium
2,1649758,COVID19,PFIZER\BIONTECH,33.0,F,2021,Chills; Fever; Headache; Nausea; Vomiting; Dia...,"['Chills', 'Diarrhoea', 'Headache', 'Irritabil...",7,284,Long
3,1129427,COVID19,PFIZER\BIONTECH,50.0,M,2021,"Pt received vaccine on Friday, March 19th. On...","['Cardio-respiratory arrest', 'Death', 'Dyspno...",4,41,Short
4,2121131,COVID19,PFIZER\BIONTECH,43.0,M,2022,Patient came into pharmacy on 2/17/22 stating ...,"['Burning sensation', 'Joint range of motion d...",5,277,Long


In [96]:
example = """
[
  {"Erythema": ["redness in neck"]},
  {
    "Pain in extremity": ["sore arm"],
    "Pruritus": ["itchy feeling"]
  },
  {"Swelling": ["mild arm swelling"]}
]
"""

input_template_mention = """
Ignore previous conversations.

Clinical Notes: {text}

Task:

1. First, determine if there is any temporal progression **across different symptoms** in the clinical notes.
   - Temporal progression means: one symptom appears first, followed by another symptom later.
   - If only the severity, intensity, or condition of a **single symptom** changes over time (e.g., pain worsens or improves), **do not** consider it a progression across different symptoms.

2. If no temporal progression across different symptoms is found:
   - Return: "No temporal information found"

3. If temporal progression **across different symptoms** is found:
   - Use the adverse effect list below:
     {suggest}
   - Reorder the list based on the sequence implied in the clinical notes.
   - If several adverse effects are mentioned **together without a clear temporal order**, group them into the same block (inside a single dictionary).
   - Do **not skip any terms** from the list — include all terms in the output.
   - For each term, extract the closest matching phrase from the clinical notes; if a term is not mentioned, assign "none" as its value.

Important Rules:
- Mention each symptom **only once** — at the first time it appears or becomes relevant.
- Do **NOT** treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times).
- If multiple symptoms are mentioned across multiple sentences **without clear timeline separation**, **group them together**.
- Only separate symptoms into different blocks when the clinical notes clearly describe a **sequence or timing** between them.

Finally, return your answer **only** in valid JSON format —
using a **list of dictionaries** to represent temporal progression and grouping, as shown below:
{example}
"""

In [129]:
# chatgpt results
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

def llm(prompt):
    try:
        response = client.chat.completions.create(
            model="o4-mini",
            messages=[{"role": "user", "content": prompt}],
            # max_completion_tokens=300,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def clean_and_parse_json(text):
    try:
        if isinstance(text, str):
            cleaned = text.replace('\n', '')
            return json.loads(cleaned)
    except json.JSONDecodeError:
        return None

    return None

def validate_output_symptoms(row):
    if isinstance(row['llm_result_cleaned'], list):
        output_symptoms = [key for dic in row['llm_result_cleaned'] for key in dic.keys()]
        input_symptoms = ast.literal_eval(row['symptom_list'])
        return set(output_symptoms) == set(input_symptoms)
    return False

def get_ordered_symptoms_flatten(symptom_list):
    symptoms = []
    if isinstance(symptom_list, list):
        for sym_dict in symptom_list:
            symptoms.extend(list(sym_dict.keys()))
    return symptoms

def get_ordered_symptoms(symptom_list):
    symptoms = []
    if isinstance(symptom_list, list):
        for sym_dict in symptom_list:
            symptoms.append(list(sym_dict.keys()))
    return symptoms

In [98]:
sampled_data['llm_result'] = sampled_data.apply(lambda row: llm(input_template_mention.format(
    text=row['symptom_text'],
    suggest=row['symptom_list'],
    example=example)
), axis=1)

In [130]:
sampled_data['llm_result_cleaned'] = sampled_data['llm_result'].apply(clean_and_parse_json)

In [131]:
sampled_data['is_symptom_list_correct'] = sampled_data.apply(validate_output_symptoms, axis=1)

In [132]:
sampled_data['ordered_symptoms'] = sampled_data['llm_result_cleaned'].apply(get_ordered_symptoms)

In [133]:
sampled_data['ordered_symptoms_flatten'] = sampled_data['llm_result_cleaned'].apply(get_ordered_symptoms_flatten)

In [108]:
sampled_data[sampled_data['is_symptom_list_correct'] == False]

,vaers_id,vax_type,vax_manu,age_yrs,sex,vax_year,symptom_text,symptom_list,num_symptoms,report_length,length_category,llm_result,llm_result_cleaned,is_symptom_list_correct
2,1649758,COVID19,PFIZER\BIONTECH,33.0,F,2021,Chills; Fever; Headache; Nausea; Vomiting; Dia...,"['Chills', 'Diarrhoea', 'Headache', 'Irritabil...",7,284,Long,No temporal information found,None,False
11,1386388,COVID19,PFIZER\BIONTECH,28.0,F,2021,Back breaking pain for three days straight Ext...,"['Abdominal pain', 'Back pain', 'Dyspnoea', 'F...",11,39,Short,No temporal information found,None,False
14,2226392,COVID19,PFIZER\BIONTECH,82.0,F,2022,"Case was vaccinate and boosted against Covid, ...","['Asthenia', 'COVID-19', 'Confusional state', ...",6,34,Short,No temporal information found,None,False
17,1065913,COVID19,PFIZER\BIONTECH,91.0,F,2021,she couldn't urine/She couldn't urinate anymor...,"['Arthralgia', 'Asthenia', 'Body temperature',...",26,1552,Extremely Long,"```json\n[\n {\n ""Feeling abnormal"": [""ver...",None,False
22,1483363,COVID19,PFIZER\BIONTECH,NaN,U,2021,Extremely allergic itching reaction; Whole ski...,"['Drug hypersensitivity', 'Erythema', 'Pruritu...",4,161,Medium,No temporal information found,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,2545912,COVID19,JANSSEN,50.0,F,2022,Assessment and Plan Patient is a 52 y.o. femal...,"['Acute respiratory failure', 'C-reactive prot...",11,197,Medium,No temporal information found,None,False
293,1637111,COVID19,JANSSEN,53.0,F,2021,"Shingles. Rash on right side of upper back, ve...","['Arthralgia', 'Herpes zoster', 'Influenza lik...",7,31,Short,No temporal information found,None,False
294,2327025,COVID19,JANSSEN,37.0,M,2022,"Extreme fatigue, body aches, chills, nasal con...","['Ageusia', 'Anosmia', 'Chills', 'Cough', 'Dia...",15,53,Short,No temporal information found,None,False
295,1261130,COVID19,JANSSEN,NaN,U,2021,TREMORS; FEVER; MUSCLE PAIN; This spontaneous ...,"['Body temperature', 'Myalgia', 'Pyrexia', 'Tr...",4,133,Medium,No temporal information found,None,False


In [135]:
sampled_data['is_symptom_list_correct'].value_counts()

is_symptom_list_correct
True     199
False    101
Name: count, dtype: int64

In [143]:
sampled_data['llm_result'][sampled_data['is_symptom_list_correct'] == False].value_counts()

llm_result
No temporal information found                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [146]:
def check_length(row):
    ordered_symptoms_list = len(row['ordered_symptoms_flatten'])
    symptom_list = len(ast.literal_eval(row['symptom_list']))
    return ordered_symptoms_list == symptom_list

sampled_data['is_same_length'] = sampled_data.apply(check_length, axis=1)

In [147]:
sampled_data.to_excel('data/sampled_annotated_data.xlsx', index=False)